# Notebook 2 — Model Training: 5 Base Models + Stacking Ensemble

**Project:** Early warning of low dissolved oxygen (DO) in rivers using multimodal data
**Course:** 21CSC305P Machine Learning — SRM Institute of Science and Technology, Ramapuram

---

This notebook trains the whole system from the processed data, end to end:

| Step | What is trained | Family |
|---|---|---|
| 1 | **Random Forest** | Machine Learning — bagging |
| 2 | **XGBoost** | Machine Learning — gradient boosting |
| 3 | **LightGBM** | Machine Learning — gradient boosting |
| 4 | **BiLSTM** | Deep Learning — recurrent network (GPU) |
| 5 | **CNN-LSTM** | Deep Learning — convolution + recurrent network (GPU) |
| 6 | **Stacking ensemble** — a meta-learner that combines the 5 models | Ensemble |
| 7 | **Early-warning rule** — threshold τ on P(Low) | Decision layer |

**Protocol (chronological, no leakage):**
* base models are fitted on **2023**;
* the **2024** validation year is used for early stopping, to fit the stacking meta-learner and to choose τ;
* everything is then scored once on the **2025** test year, which no step above has seen.

Every hyperparameter is read from `config.yaml` — the same values frozen in `FROZEN.md`.
Trained models are written to **`models/notebook_run/`**, so the frozen models used by the app are never touched.
The full evaluation and comparison with existing models is in **Notebook 1**.

### ▶ Running on Google Colab
Put **`ML_MAIN_V2_colab.zip`** (made by `python make_colab_zip.py` on the laptop) in the top folder of your Google Drive,
then run the cell below. On your own computer it does nothing.

In [ ]:
import os, sys, zipfile
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    ZIP = "/content/drive/MyDrive/ML_MAIN_V2_colab.zip"
    if not os.path.exists("/content/ML_MAIN_V2/config.yaml"):
        print("unzipping the project (about 1 minute) ...")
        zipfile.ZipFile(ZIP).extractall("/content")
    os.chdir("/content/ML_MAIN_V2/notebooks")
    print("project ready in /content/ML_MAIN_V2")
else:
    print("Not running on Colab - nothing to do.")

## 0. Settings

| Setting | Full run (default) | Quick demo |
|---|---|---|
| `HORIZONS_TO_TRAIN` | `[6, 12, 24, 48]` — about **60–90 min** on the RTX 3050 laptop | `[24]` — about **15–20 min** |
| `DL_SEEDS` | `[42]` — one network per architecture | `[42, 43, 44]` reproduces the frozen system (3 seeds averaged, +15 min) |
| `SMOKE_TEST` | `False` | `True` — 3 stations, few trees, 1 epoch: checks the pipeline in ~2 min (scores are meaningless) |
| `RUN_MODE` | `"train"` — trains every model from scratch | `"load"` — loads the models a previous `"train"` run saved in `models/notebook_run/` and repeats every prediction and score: **about 3 min for `[24]`, 8–10 min for all horizons** on Colab |

Random Forest is the slowest step (≈ 5 min per horizon and task); the deep models run on the GPU.
**For a presentation:** run once with `"train"` on the laptop and save the notebook — the outputs stay inside the file
and show instantly in Colab without running anything. Use `"load"` if you are asked to run it live.

In [ ]:
HORIZONS_TO_TRAIN = [6, 12, 24, 48]   # [24] for a quick demo
DL_SEEDS = [42]                       # [42, 43, 44] = the frozen system
SMOKE_TEST = False                    # True = tiny pipeline check only
SAVE_MODELS = True                    # write the trained models to models/notebook_run/
RUN_MODE = "train"                    # "train" = fit everything | "load" = reuse the models saved by a "train" run

In [ ]:
import os, sys, json, time, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")   # deterministic cuBLAS, set before torch

ROOT = Path.cwd()
while not (ROOT / "config.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from IPython.display import display, Markdown

import sklearn, xgboost as xgb, lightgbm as lgb, torch
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge, LogisticRegression

from src.config import load_config, set_seeds
from src.models import common as C
from src.evaluation.metrics import (regression_metrics, classification_metrics,
                                    early_warning_metrics, to_class)

CFG = load_config()
SEED = CFG["seed"]
ALL_HORIZONS = CFG["horizons"]
CLASS_NAMES = CFG["target"]["class_names"]
DO = CFG["target"]["column"]
M = CFG["models"]
set_seeds(SEED)

OUT_MODELS = ROOT / "models" / "notebook_run"
OUT_RESULTS = ROOT / "results" / "notebook_run"
for p in [OUT_MODELS, OUT_RESULTS]:
    p.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .3,
                     "axes.spines.top": False, "axes.spines.right": False})
pd.set_option("display.precision", 4)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"python {sys.version.split()[0]} | scikit-learn {sklearn.__version__} | xgboost {xgb.__version__} | "
      f"lightgbm {lgb.__version__} | torch {torch.__version__}")
print("deep-learning device:", device, f"({torch.cuda.get_device_name(0)})" if device.type == "cuda" else "")
print(f"horizons: {HORIZONS_TO_TRAIN} | DL seeds: {DL_SEEDS} | smoke test: {SMOKE_TEST}")
TIMES = {}          # training time of every model, for the summary
assert RUN_MODE in ("train", "load"), RUN_MODE
LOAD = RUN_MODE == "load"
LOG_FILE = OUT_MODELS / "training_log.joblib"     # curves and times of the "train" run, reused in "load" mode
if LOAD:
    if not LOG_FILE.exists():
        raise FileNotFoundError("RUN_MODE='load' needs a finished RUN_MODE='train' run: models/notebook_run/ is empty.")
    LOG = joblib.load(LOG_FILE)
    missing = [h for h in HORIZONS_TO_TRAIN if h not in LOG["horizons"]]
    assert not missing, f"the saved run has no models for horizons {missing}; it trained {LOG['horizons']}"
    print(f"LOAD mode: reusing the models trained on {LOG['trained_on']} (horizons {LOG['horizons']}, seeds {LOG['seeds']})")
    DL_SEEDS = LOG["seeds"]
else:
    LOG = {"horizons": list(HORIZONS_TO_TRAIN), "seeds": list(DL_SEEDS), "times": {}, "xgb": {}, "lgb": {}, "dl": {},
           "trained_on": time.strftime("%Y-%m-%d %H:%M")}
def save_log():
    if not LOAD and SAVE_MODELS:
        LOG["times"] = dict(TIMES); joblib.dump(LOG, LOG_FILE)
hh = 24 if 24 in HORIZONS_TO_TRAIN else HORIZONS_TO_TRAIN[-1]   # the horizon used for plots

## 1. Load the processed data

The processed tables were built by `src/features/build_features.py`: hourly rows per station with **211 features**
built only from data at or before time *t*, and targets `y_reg_h` (DO in mg/L at *t+h*) and `y_cls_h`
(0 = Low, 1 = Medium, 2 = Good).

In [ ]:
FEATURES = C.feature_columns()
train = C.load_split("train")       # 2023
val = C.load_split("val")           # 2024
test = C.load_split("test")         # 2025

SMOKE_STATIONS = None
if SMOKE_TEST:
    SMOKE_STATIONS = sorted(train["station"].unique())[:3]
    train, val, test = [d[d["station"].isin(SMOKE_STATIONS)].reset_index(drop=True) for d in (train, val, test)]

for name, d in [("train (2023)", train), ("validation (2024)", val), ("test (2025)", test)]:
    print(f"{name:<18} {len(d):>8,} rows | {d['station'].nunique()} stations | "
          f"{d['time'].min():%Y-%m-%d} .. {d['time'].max():%Y-%m-%d}")
print(f"\n{len(FEATURES)} input features")
with open(ROOT / CFG["paths"]["feature_groups"]) as f:
    groups = json.load(f)
display(pd.DataFrame({"group": g, "features": len(groups[g]), "examples": ", ".join(groups[g][:4]) + " ..."}
                     for g in ["water", "weather", "air", "time", "station"]).set_index("group"))

In [ ]:
def xy(df, task, h):
    # features and target; rows with a missing target are dropped (nothing to learn from them)
    y = pd.to_numeric(df[C.target_name(task, h)], errors="coerce")
    ok = y.notna().to_numpy()
    y = y[ok].astype(int) if task == "classification" else y[ok]
    return df.loc[ok, FEATURES], y

def y_true(df, task, h):
    return pd.to_numeric(df[C.target_name(task, h)], errors="coerce").to_numpy(dtype="float64")

PERSIST = {"val": val[DO].to_numpy(dtype="float64"), "test": test[DO].to_numpy(dtype="float64")}
CLASS_NOW = {"val": to_class(val[DO]), "test": to_class(test[DO])}

def score(split, h, reg, proba):
    # regression + classification + early-warning metrics of one model on one split
    df = val if split == "val" else test
    r = regression_metrics(y_true(df, "regression", h), reg, PERSIST[split])
    c = classification_metrics(y_true(df, "classification", h), proba.argmax(1), proba)
    e = early_warning_metrics(CLASS_NOW[split], y_true(df, "classification", h), proba.argmax(1))
    return {"R2": r["r2"], "RMSE": r["rmse"], "MAE": r["mae"], "skill": r["skill_vs_persistence"],
            "accuracy": c["accuracy"], "macro_F1": c["f1_macro"], "ROC_AUC": c["roc_auc_ovr_macro"],
            "Low_recall": c["recall_Low"], "onset_recall": e["onset_recall"]}

# PRED[model][h] = {"val_reg", "val_proba", "test_reg", "test_proba"} — collected for the ensemble
PRED = {m: {} for m in ["random_forest", "xgboost", "lightgbm", "bilstm", "cnn_lstm"]}
ROWS = []          # per-model scores

## 2. Base model 1 — Random Forest

* 300 trees, each grown on a random half of the rows (`max_samples=0.5`) and 30 % of the features per split.
* `min_samples_leaf=20` keeps leaves from memorising single hours.
* Random Forest cannot handle missing values, so a **median imputer fitted on 2023 only** sits in the same pipeline.
* Classification uses `class_weight="balanced_subsample"` so the minority **Low** class is not ignored.
* No early stopping — a forest does not overfit by adding trees.

In [ ]:
rf_cfg = dict(M["random_forest"]); rf_cw = rf_cfg.pop("classifier_class_weight")
if SMOKE_TEST: rf_cfg["n_estimators"] = 20
print("Random Forest settings:", rf_cfg, "| classifier class_weight:", rf_cw)

for h in HORIZONS_TO_TRAIN:
    out = {}
    for task in ["regression", "classification"]:
        X, y = xy(train, task, h)
        path = OUT_MODELS / f"random_forest_{task}_h{h}.joblib"
        if LOAD:
            model = joblib.load(path); secs = LOG["times"].get(f"random_forest_{task}_h{h}", np.nan)
        else:
            rf = (RandomForestRegressor(random_state=SEED, **rf_cfg) if task == "regression"
                  else RandomForestClassifier(random_state=SEED, class_weight=rf_cw, **rf_cfg))
            model = Pipeline([("impute", SimpleImputer(strategy="median")), ("model", rf)])
            t0 = time.perf_counter(); model.fit(X, y); secs = time.perf_counter() - t0
        TIMES[f"random_forest_{task}_h{h}"] = secs
        for split, df in [("val", val), ("test", test)]:
            out[f"{split}_{'reg' if task == 'regression' else 'proba'}"] = (
                model.predict(df[FEATURES]) if task == "regression" else model.predict_proba(df[FEATURES]))
        if SAVE_MODELS and not LOAD:
            joblib.dump(model, path, compress=3)
        print(f"  h={h:>2} {task:<14} {'loaded; was ' if LOAD else ''}trained on {len(X):,} rows in {secs:6.1f}s")
        if task == "regression" and h == hh:
            RF_IMPORTANCE = pd.Series(model.named_steps["model"].feature_importances_, index=FEATURES)
    PRED["random_forest"][h] = out
    for split in ["val", "test"]:
        ROWS.append({"model": "random_forest", "horizon": h, "split": split,
                     **score(split, h, out[f"{split}_reg"], out[f"{split}_proba"])})
del model
save_log()
display(pd.DataFrame(ROWS).query("model == 'random_forest'").set_index(["horizon", "split"]).drop(columns="model"))

## 3. Base model 2 — XGBoost

* Gradient boosting with second-order (Newton) updates and histogram splits; learning rate 0.05, depth 8,
  80 % row and 70 % column subsampling.
* **Missing values handled natively** — XGBoost learns which branch a missing value should take.
* **Early stopping on 2024:** up to 2,000 trees, stopping when the validation loss has not improved for 50 rounds.
* Classes balanced with sample weights *w = n / (3 · count of class)*.

In [ ]:
xgb_cfg = dict(M["xgboost"])
max_rounds, es_rounds = (60, 10) if SMOKE_TEST else (M["max_rounds"], M["early_stopping_rounds"])
print("XGBoost settings:", xgb_cfg, f"| max {max_rounds} rounds, early stopping {es_rounds}")
XGB_CURVES = {}

for h in HORIZONS_TO_TRAIN:
    out = {}
    for task in ["regression", "classification"]:
        X, y = xy(train, task, h)
        Xv, yv = xy(val, task, h)
        common = dict(n_estimators=max_rounds, early_stopping_rounds=es_rounds, random_state=SEED, **xgb_cfg)
        if task == "regression":
            model, w = xgb.XGBRegressor(eval_metric="rmse", **common), None
        else:
            model = xgb.XGBClassifier(objective="multi:softprob", num_class=3, eval_metric="mlogloss", **common)
            w = C.balanced_sample_weight(y)
        path = OUT_MODELS / f"xgboost_{task}_h{h}.json"
        if LOAD:
            model.load_model(path); secs = LOG["times"].get(f"xgboost_{task}_h{h}", np.nan)
            XGB_CURVES[(task, h)] = LOG["xgb"][(task, h)]
        else:
            t0 = time.perf_counter()
            model.fit(X, y, sample_weight=w, eval_set=[(X, y), (Xv, yv)], verbose=False)
            secs = time.perf_counter() - t0
            XGB_CURVES[(task, h)] = LOG["xgb"][(task, h)] = model.evals_result()
        TIMES[f"xgboost_{task}_h{h}"] = secs
        for split, df in [("val", val), ("test", test)]:
            out[f"{split}_{'reg' if task == 'regression' else 'proba'}"] = (
                model.predict(df[FEATURES]) if task == "regression" else model.predict_proba(df[FEATURES]))
        if SAVE_MODELS and not LOAD:
            model.save_model(path)
        print(f"  h={h:>2} {task:<14} best iteration {model.best_iteration:>4} | {secs:6.1f}s")
    PRED["xgboost"][h] = out
    for split in ["val", "test"]:
        ROWS.append({"model": "xgboost", "horizon": h, "split": split,
                     **score(split, h, out[f"{split}_reg"], out[f"{split}_proba"])})
save_log()
display(pd.DataFrame(ROWS).query("model == 'xgboost'").set_index(["horizon", "split"]).drop(columns="model"))

## 4. Base model 3 — LightGBM

* Leaf-wise gradient boosting (63 leaves, ≥ 50 rows per leaf), learning rate 0.05, 70 % feature and 80 % row bagging.
* `station_id` is given as a true **categorical** feature, so each river can get its own splits.
* Native missing-value handling, `class_weight="balanced"`, early stopping on 2024 exactly as for XGBoost.

In [ ]:
lgb_cfg = dict(M["lightgbm"]); cat_cols = lgb_cfg.pop("categorical_features"); lgb_cw = lgb_cfg.pop("classifier_class_weight")
print("LightGBM settings:", lgb_cfg, "| categorical:", cat_cols)
CATS = {c: sorted(train[c].astype("int32").unique()) for c in cat_cols}

def as_lgb(X):
    X = X.copy()
    for c, levels in CATS.items():
        X[c] = pd.Categorical(X[c].astype("int32"), categories=levels)
    return X

LGB_CURVES = {}
for h in HORIZONS_TO_TRAIN:
    out = {}
    for task in ["regression", "classification"]:
        X, y = xy(train, task, h)
        Xv, yv = xy(val, task, h)
        common = dict(n_estimators=max_rounds, random_state=SEED, verbose=-1, **lgb_cfg)
        model = (lgb.LGBMRegressor(objective="regression", **common) if task == "regression"
                 else lgb.LGBMClassifier(objective="multiclass", num_class=3, class_weight=lgb_cw, **common))
        path = OUT_MODELS / f"lightgbm_{task}_h{h}.joblib"
        if LOAD:
            model = joblib.load(path)["model"]; secs = LOG["times"].get(f"lightgbm_{task}_h{h}", np.nan)
            curve = LOG["lgb"][(task, h)]
        else:
            curve = {}
            t0 = time.perf_counter()
            model.fit(as_lgb(X), y, eval_set=[(as_lgb(X), y), (as_lgb(Xv), yv)], eval_names=["train", "validation"],
                      callbacks=[lgb.early_stopping(es_rounds, verbose=False), lgb.log_evaluation(0),
                                 lgb.record_evaluation(curve)])
            secs = time.perf_counter() - t0
            LOG["lgb"][(task, h)] = curve
        TIMES[f"lightgbm_{task}_h{h}"] = secs
        LGB_CURVES[(task, h)] = curve
        for split, df in [("val", val), ("test", test)]:
            Xs = as_lgb(df[FEATURES])
            out[f"{split}_{'reg' if task == 'regression' else 'proba'}"] = (
                model.predict(Xs) if task == "regression" else model.predict_proba(Xs))
        if SAVE_MODELS and not LOAD:
            joblib.dump({"model": model, "cats": CATS}, path, compress=3)
        if task == "regression" and h == hh:
            LGB_IMPORTANCE = pd.Series(model.booster_.feature_importance("gain"), index=FEATURES)
        print(f"  h={h:>2} {task:<14} best iteration {model.best_iteration_:>4} | {secs:6.1f}s")
    PRED["lightgbm"][h] = out
    for split in ["val", "test"]:
        ROWS.append({"model": "lightgbm", "horizon": h, "split": split,
                     **score(split, h, out[f"{split}_reg"], out[f"{split}_proba"])})
save_log()
display(pd.DataFrame(ROWS).query("model == 'lightgbm'").set_index(["horizon", "split"]).drop(columns="model"))

### Training curves of the boosted trees and what the trees rely on

The curves show the training and validation loss per boosting round; the dashed line is the round early stopping
kept. Feature importance shows which inputs drive the 24 h forecast.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 3.6))
for ax, (lib, task) in zip(axes, [("XGBoost", "regression"), ("XGBoost", "classification"),
                                  ("LightGBM", "regression"), ("LightGBM", "classification")]):
    if lib == "XGBoost":
        ev = XGB_CURVES[(task, hh)]; metric = list(ev["validation_0"])[0]
        tr, va = ev["validation_0"][metric], ev["validation_1"][metric]
    else:
        ev = LGB_CURVES[(task, hh)]; metric = list(ev["validation"])[0]
        tr, va = ev["train"][metric], ev["validation"][metric]
    best = int(np.argmin(va))
    ax.plot(tr, label="train (2023)"); ax.plot(va, label="validation (2024)")
    ax.axvline(best, color="k", ls="--", lw=1)
    ax.set_title(f"{lib} {task[:5]}., {hh} h ({metric})", fontsize=10); ax.set_xlabel("boosting round")
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(OUT_RESULTS / "boosting_curves.png", bbox_inches="tight"); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, (imp, title) in zip(axes, [(RF_IMPORTANCE, "Random Forest (impurity)"), (LGB_IMPORTANCE, "LightGBM (gain)")]):
    top = (imp / imp.sum()).sort_values().tail(15)
    grp = {f: g for g in ["water", "weather", "air", "time", "station"] for f in groups[g]}
    colors = {"water": "#2171b5", "weather": "#41ab5d", "air": "#fe9929", "time": "#969696", "station": "#807dba"}
    ax.barh(top.index, top.values, color=[colors[grp[f]] for f in top.index])
    ax.set_title(f"Top 15 features — {title}, regression {hh} h"); ax.set_xlabel("share of importance")
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in colors.values()]
axes[1].legend(handles, colors.keys(), loc="lower right")
plt.tight_layout(); plt.savefig(OUT_RESULTS / "feature_importance.png", bbox_inches="tight"); plt.show()

## 5–6. Base models 4 and 5 — the deep networks (BiLSTM and CNN-LSTM)

**Input.** For a forecast at time *t*, the network reads the last **72 hours** of **29 raw hourly channels**
(6 water variables + their 6 "was missing" flags, DO saturation, 8 weather, 8 air-quality, 4 time encodings),
plus a learned **station embedding** (8 numbers per river) and a **hybrid branch** with the 211 engineered features at *t*.
Missing values are forward-filled within a station (never from the future); scalers are fitted on 2023 only.

**Multi-task output.** One network predicts everything in one pass: 4 DO values (one per horizon) and
4 × 3 class probabilities. The regression head predicts the **change** in DO, so the network starts from
"no change" instead of having to learn every river's level.

**Loss.** Masked MSE on the scaled DO change + class-weighted cross-entropy, averaged over horizons.

**Training.** AdamW (lr 5·10⁻⁴, weight decay 10⁻⁴), dropout 0.3, ReduceLROnPlateau, gradient clipping 1.0,
mixed precision on the GPU, **early stopping on the 2024 validation loss** (patience 4).
These settings are the variant chosen in step 5 by a 10-run search (`reg_hybrid_72h_small`).

In [ ]:
from src.models import sequences as S
from src.models import train_dl as T
from src.models.dl_models import MODELS as DL_MODELS, count_parameters

settings = T.variant_settings(CFG["final_test"].get("dl_variant", "reg_hybrid_72h_small"))   # the frozen variant
seq_len = settings["sequence_length"]
print("variant:", settings["variant"], "| window:", seq_len, "h | channels:", S.N_CHANNELS,
      "| engineered features:", settings["use_engineered_features"])

t0 = time.perf_counter()
if LOAD:     # the scalers fitted on 2023 during the "train" run travel with the models
    sc = joblib.load(OUT_MODELS / "dl_scaler.joblib")
    kw = dict(scaler=sc["scaler"], medians=sc["medians"], stations=SMOKE_STATIONS, seq_len=seq_len, with_features=True,
              feature_scaler=sc["feature_scaler"], feature_medians=sc["feature_medians"])
    seq_train = None
    do_scale = sc["do_scale"]
else:
    seq_train = S.prepare("train", stations=SMOKE_STATIONS, seq_len=seq_len, with_features=True)
    kw = dict(scaler=seq_train.scaler, medians=seq_train.medians, stations=SMOKE_STATIONS, seq_len=seq_len,
              with_features=True, feature_scaler=seq_train.feature_scaler, feature_medians=seq_train.feature_medians)
    do_scale = float(seq_train.scaler.scale_[S.CHANNELS.index("do_mgl")])
seq_val, seq_test = S.prepare("val", **kw), S.prepare("test", **kw)
splits_seq = [("val", seq_val), ("test", seq_test)] + ([] if LOAD else [("train", seq_train)])
targets = {n: T.build_targets(d, do_scale) for n, d in splits_seq}
print(f"windows: " + " | ".join(f"{n} {len(d):,}" for n, d in splits_seq) + f" | built in {time.perf_counter() - t0:.0f}s")
print("channels:", S.CHANNELS)

n_stations = int(max(max(d.station_ids.values()) for _, d in splits_seq) + 1)
N_FEAT = seq_val.features.shape[1]
for name in ["bilstm", "cnn_lstm"]:
    net = DL_MODELS[name](S.N_CHANNELS, n_stations, cfg=T.architecture_overrides(settings, name),
                          n_features=N_FEAT)
    display(Markdown(f"**{name}** — {count_parameters(net):,} trainable parameters"))
    print(net)

In [ ]:
def align(seq, df, arr):
    # put the network's rows back in the order of the processed table (trees and networks score identical rows)
    k = pd.MultiIndex.from_frame(seq.meta[["station", "time"]])
    target = pd.MultiIndex.from_frame(df[["station", "time"]])
    pos = k.get_indexer(target)
    out = np.full((len(df),) + arr.shape[1:], np.nan, dtype="float64")
    ok = pos >= 0
    out[ok] = arr[pos[ok]]
    return out

DL_HISTORY = {}
max_epochs = 1 if SMOKE_TEST else None
for name in ["bilstm", "cnn_lstm"]:
    regs = {"val": [], "test": []}; probs = {"val": [], "test": []}
    t_model = time.perf_counter()
    for seed in DL_SEEDS:
        print(f"\n=== {name}, seed {seed} ===")
        if LOAD:
            from src.models.dl_models import MultiTaskLoss
            ckpt = torch.load(OUT_MODELS / f"{name}_seed{seed}.pt", map_location=device)
            model = DL_MODELS[name](S.N_CHANNELS, n_stations, cfg=ckpt["arch"], n_features=N_FEAT).to(device)
            model.load_state_dict(ckpt["state_dict"]); model.eval()
            loss_fn = MultiTaskLoss(T.class_weights_from_train(targets["val"][1])).to(device)
            bs = settings["batch_size"]
            hist = LOG["dl"][(name, seed)]
            print(f"  loaded (trained for {int(hist['epoch'].max())} epochs, kept epoch {int(hist.loc[hist['val_loss'].idxmin(), 'epoch'])})")
        else:
            model, hist, best, bs, loss_fn = T.train_model(
                name, settings, seq_train, seq_val, *targets["train"], *targets["val"],
                do_scale, device, seed, max_epochs=max_epochs, verbose=True)
            LOG["dl"][(name, seed)] = hist
            print(f"  kept epoch {best['epoch']} (lowest validation loss {best['val_loss']:.4f})")
        DL_HISTORY[(name, seed)] = hist
        for split, seq in [("val", seq_val), ("test", seq_test)]:
            _, do_pred, cls_prob = T.predict(model, seq, *targets[split], loss_fn, device, bs, do_scale)
            df = val if split == "val" else test
            regs[split].append(align(seq, df, do_pred)); probs[split].append(align(seq, df, cls_prob))
        if SAVE_MODELS and not LOAD:
            torch.save({"state_dict": model.state_dict(), "channels": S.CHANNELS, "seq_len": seq_len,
                        "do_scale": do_scale, "seed": seed, "arch": T.architecture_overrides(settings, name)},
                       OUT_MODELS / f"{name}_seed{seed}.pt")
        del model
        if device.type == "cuda": torch.cuda.empty_cache()
    TIMES[name] = LOG["times"].get(name, np.nan) if LOAD else time.perf_counter() - t_model
    # average the seeds, then split the multi-task output per horizon
    reg = {s: np.mean(regs[s], axis=0) for s in regs}; prob = {s: np.mean(probs[s], axis=0) for s in probs}
    for h in HORIZONS_TO_TRAIN:
        j = ALL_HORIZONS.index(h)
        PRED[name][h] = {"val_reg": reg["val"][:, j], "val_proba": prob["val"][:, j, :],
                         "test_reg": reg["test"][:, j], "test_proba": prob["test"][:, j, :]}
        for split in ["val", "test"]:
            ROWS.append({"model": name, "horizon": h, "split": split,
                         **score(split, h, PRED[name][h][f"{split}_reg"], PRED[name][h][f"{split}_proba"])})

save_log()
if SAVE_MODELS and not LOAD:
    joblib.dump({"scaler": seq_train.scaler, "medians": seq_train.medians, "feature_scaler": seq_train.feature_scaler,
                 "feature_medians": seq_train.feature_medians, "channels": S.CHANNELS, "seq_len": seq_len,
                 "do_scale": do_scale}, OUT_MODELS / "dl_scaler.joblib")
display(pd.DataFrame(ROWS).query("model in ['bilstm', 'cnn_lstm']").set_index(["model", "horizon", "split"]))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 3.8))
for (name, seed), hist in DL_HISTORY.items():
    c = "#756bb1" if name == "cnn_lstm" else "#9e9ac8"
    axes[0].plot(hist["epoch"], hist["train_loss"], "--", color=c)
    axes[0].plot(hist["epoch"], hist["val_loss"], "-o", color=c, label=f"{name} s{seed} (solid = validation)")
    axes[1].plot(hist["epoch"], hist["val_rmse_24"], "-o", color=c, label=f"{name} s{seed}")
    axes[2].plot(hist["epoch"], hist["val_f1_24"], "-o", color=c, label=f"{name} s{seed}")
for ax, t in zip(axes, ["loss (dashed = train, solid = validation)", "validation RMSE at 24 h (mg/L)",
                        "validation macro F1 at 24 h"]):
    ax.set_title(t, fontsize=10); ax.set_xlabel("epoch")
axes[0].legend(fontsize=7)
plt.tight_layout(); plt.savefig(OUT_RESULTS / "dl_training_curves.png", bbox_inches="tight"); plt.show()

## 7. The 5 base models side by side

Scores on the 2024 validation year (used for early stopping, so slightly optimistic) and on the untouched 2025 test year.

In [ ]:
BASE = pd.DataFrame(ROWS)
for split, label in [("test", "2025 TEST"), ("val", "2024 validation")]:
    t = BASE[BASE.split == split].pivot_table(index="model", columns="horizon", values=["RMSE", "macro_F1"])
    display(Markdown(f"**{label}** — RMSE (mg/L, lower is better) and macro F1 (higher is better)"))
    display(t.loc[["random_forest", "xgboost", "lightgbm", "bilstm", "cnn_lstm"]].style.format("{:.4f}")
            .highlight_min(subset=[c for c in t.columns if c[0] == "RMSE"], color="#ffd8b1")
            .highlight_max(subset=[c for c in t.columns if c[0] == "macro_F1"], color="#ffd8b1"))

In [ ]:
# Why combine them? Members that make DIFFERENT errors cancel each other out in an ensemble.
err = pd.DataFrame({m: PRED[m][hh]["test_reg"] - y_true(test, "regression", hh) for m in PRED}).dropna()
fig, ax = plt.subplots(figsize=(6, 4.8))
im = ax.imshow(err.corr().values, cmap="RdYlBu_r", vmin=.85, vmax=1)
ax.set_xticks(range(5)); ax.set_yticks(range(5)); ax.set_xticklabels(err.columns, rotation=30); ax.set_yticklabels(err.columns)
for i in range(5):
    for j in range(5):
        ax.text(j, i, f"{err.corr().values[i, j]:.3f}", ha="center", va="center", fontsize=9)
plt.colorbar(im); ax.set_title(f"Correlation of {hh} h errors (2025): DL vs trees is lowest")
plt.tight_layout(); plt.savefig(OUT_RESULTS / "error_correlation.png", bbox_inches="tight"); plt.show()

## 8. The stacking ensemble

A **meta-learner** is trained to combine the 5 base models. It is fitted on the **2024** predictions — data the base
models never trained on — so it learns how much to trust each model on unseen data, not on data it memorised.

* **Regression:** Ridge regression (α = 1) on the 5 DO forecasts → final DO forecast.
* **Classification:** multinomial logistic regression (balanced class weights) on the 5 × 3 = 15 class probabilities
  → final P(Low), P(Medium), P(Good).

A **simple average** of the 5 models is built too, as the naive ensemble to beat.

In [ ]:
MEMBERS = ["random_forest", "xgboost", "lightgbm", "bilstm", "cnn_lstm"]
ENS = {}            # ENS[h] = dict of final predictions
STACK_MODELS = {}
for h in HORIZONS_TO_TRAIN:
    R = {s: np.column_stack([PRED[m][h][f"{s}_reg"] for m in MEMBERS]) for s in ["val", "test"]}
    P = {s: np.stack([PRED[m][h][f"{s}_proba"] for m in MEMBERS], axis=1) for s in ["val", "test"]}
    yv_reg, yv_cls = y_true(val, "regression", h), y_true(val, "classification", h)

    ok = ~np.isnan(yv_reg) & ~np.isnan(R["val"]).any(1)
    ridge = Ridge(alpha=1.0).fit(R["val"][ok], yv_reg[ok])

    Xc = {s: P[s].reshape(len(P[s]), -1) for s in P}
    ok = ~np.isnan(yv_cls) & ~np.isnan(Xc["val"]).any(1)
    logit = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED).fit(Xc["val"][ok], yv_cls[ok].astype(int))

    def safe(f, X, width=None):   # rows with a missing member forecast get NaN instead of crashing
        out = np.full(len(X) if width is None else (len(X), width), np.nan)
        good = ~np.isnan(X).any(1)
        out[good] = f(X[good])
        return out

    ENS[h] = {}
    for s in ["val", "test"]:
        ENS[h][f"stacking_{s}_reg"] = safe(ridge.predict, R[s])
        ENS[h][f"stacking_{s}_proba"] = safe(logit.predict_proba, Xc[s], width=3)
        ENS[h][f"average_{s}_reg"] = R[s].mean(1)
        ENS[h][f"average_{s}_proba"] = P[s].mean(1)
    STACK_MODELS[h] = {"ridge": ridge, "logit": logit, "members": MEMBERS}
    if SAVE_MODELS:
        joblib.dump(STACK_MODELS[h], OUT_MODELS / f"stacking_h{h}.joblib")

display(Markdown("**What the meta-learner learned — Ridge weights on each base model's DO forecast:**"))
w = pd.DataFrame({f"{h} h": np.r_[STACK_MODELS[h]["ridge"].coef_, STACK_MODELS[h]["ridge"].intercept_]
                  for h in HORIZONS_TO_TRAIN}, index=MEMBERS + ["intercept"])
display(w.style.format("{:+.3f}").bar(subset=pd.IndexSlice[MEMBERS, :], color="#fdae6b", align="zero"))
print("The weights sum to ~1: the ensemble is a learned weighted vote, and each model's weight reflects how useful it\n"
      "was on 2024 *given the other four* - a model can earn weight by being different, not only by being accurate.")

## 9. The early-warning rule

Argmax answers *"which class is most likely?"*. A warning system must answer *"is the risk of Low high enough to act on?"*.
The rule issues a **Low warning whenever P(Low) ≥ τ**. τ is chosen on **2024** as the value that catches the most
**onsets** (safe now → Low at *t+h*) while keeping the **false-alarm rate ≤ 1.8 %** (the 2 % promise minus a safety margin).
τ is then frozen and applied unchanged to 2025.

In [ ]:
def apply_rule(proba, tau):
    pred = 1 + np.nan_to_num(proba[:, 1:], nan=-1).argmax(axis=1)   # best of Medium / Good ...
    pred[proba[:, 0] >= tau] = 0                                       # ... unless P(Low) >= tau
    return pred

WR = CFG["warning_rule"]
BUDGET = 0.018
def choose_tau(proba, split="val", h=24):
    df = val if split == "val" else test
    best = None
    for tau in np.arange(WR["tau_grid_start"], WR["tau_grid_stop"] + 1e-9, WR["tau_grid_step"]):
        e = early_warning_metrics(CLASS_NOW[split], y_true(df, "classification", h), apply_rule(proba, tau))
        if np.isnan(e["false_alarm_rate"]) or e["false_alarm_rate"] > BUDGET:
            continue
        key = (e["onset_recall"], -e["false_alarm_rate"], tau)
        if best is None or key > best[0]:
            best = (key, round(float(tau), 2), e["onset_recall"], e["false_alarm_rate"])
    return best

TAU = {}
rows = []
for h in HORIZONS_TO_TRAIN:
    b = choose_tau(ENS[h]["stacking_val_proba"], "val", h)
    TAU[h] = b[1] if b else None
    rows.append({"horizon (h)": h, "τ chosen on 2024": TAU[h],
                 "2024 onset recall": b[2] if b else np.nan, "2024 false-alarm rate": b[3] if b else np.nan})
display(pd.DataFrame(rows).set_index("horizon (h)").style.format("{:.3f}"))
if SAVE_MODELS:
    json.dump({str(h): TAU[h] for h in TAU}, open(OUT_MODELS / "warning_thresholds.json", "w"), indent=2)

## 10. Final evaluation on the 2025 test year

The complete system — 5 base models → stacking meta-learner → warning rule — against each base model and the simple
average. Nothing here was fitted or tuned on 2025.

In [ ]:
def full_score(h, reg, proba, rule_tau=None):
    yr, yc = y_true(test, "regression", h), y_true(test, "classification", h)
    pred = proba.argmax(1) if rule_tau is None else apply_rule(proba, rule_tau)
    r = regression_metrics(yr, reg, PERSIST["test"])
    c = classification_metrics(yc, pred, proba)
    e = early_warning_metrics(CLASS_NOW["test"], yc, pred)
    return {"R2": r["r2"], "RMSE": r["rmse"], "MAE": r["mae"], "skill vs persistence": r["skill_vs_persistence"],
            "accuracy": c["accuracy"], "balanced acc": c["balanced_accuracy"], "precision": c["precision_macro"],
            "recall": c["recall_macro"], "macro F1": c["f1_macro"], "ROC-AUC": c["roc_auc_ovr_macro"],
            "Low recall": c["recall_Low"], "onset recall": e["onset_recall"], "false-alarm rate": e["false_alarm_rate"]}

def persistence_score(h):
    yr, yc = y_true(test, "regression", h), y_true(test, "classification", h)
    r = regression_metrics(yr, PERSIST["test"], PERSIST["test"])
    c = classification_metrics(yc, CLASS_NOW["test"])
    e = early_warning_metrics(CLASS_NOW["test"], yc, CLASS_NOW["test"])
    return {"R2": r["r2"], "RMSE": r["rmse"], "MAE": r["mae"], "skill vs persistence": 0.0,
            "accuracy": c["accuracy"], "balanced acc": c["balanced_accuracy"], "precision": c["precision_macro"],
            "recall": c["recall_macro"], "macro F1": c["f1_macro"], "ROC-AUC": np.nan, "Low recall": c["recall_Low"],
            "onset recall": e["onset_recall"], "false-alarm rate": e["false_alarm_rate"]}

FINAL = []
for h in HORIZONS_TO_TRAIN:
    FINAL.append({"method": "persistence (existing)", "horizon": h, **persistence_score(h)})
    for m in MEMBERS:
        FINAL.append({"method": m, "horizon": h, **full_score(h, PRED[m][h]["test_reg"], PRED[m][h]["test_proba"])})
    FINAL.append({"method": "simple average", "horizon": h,
                  **full_score(h, ENS[h]["average_test_reg"], ENS[h]["average_test_proba"])})
    FINAL.append({"method": "stacking (argmax)", "horizon": h,
                  **full_score(h, ENS[h]["stacking_test_reg"], ENS[h]["stacking_test_proba"])})
    FINAL.append({"method": "STACKING + WARNING RULE (proposed)", "horizon": h,
                  **full_score(h, ENS[h]["stacking_test_reg"], ENS[h]["stacking_test_proba"], TAU[h])})
FINAL = pd.DataFrame(FINAL)
FINAL.to_csv(OUT_RESULTS / "final_metrics_2025.csv", index=False)

for h in HORIZONS_TO_TRAIN:
    t = FINAL[FINAL.horizon == h].set_index("method").drop(columns="horizon")
    display(Markdown(f"### {h} hours ahead — 2025 test year"))
    display(t.style.format("{:.4f}", na_rep="–")
            .highlight_min(subset=["RMSE", "MAE"], color="#ffd8b1")
            .highlight_max(subset=["R2", "macro F1", "Low recall", "onset recall"], color="#ffd8b1")
            .apply(lambda s: ["font-weight:bold;background-color:#fff1e6" if "proposed" in s.name else "" for _ in s], axis=1))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.2))
order = ["random_forest", "xgboost", "lightgbm", "bilstm", "cnn_lstm", "simple average", "STACKING + WARNING RULE (proposed)"]
col = ["#6baed6", "#4292c6", "#2171b5", "#9e9ac8", "#756bb1", "#fdae6b", "#d94801"]
for m, c in zip(order, col):
    s = FINAL[FINAL.method == m]
    axes[0].plot(s["horizon"], s["RMSE"], "-o", color=c, lw=3 if "proposed" in m else 1.3, label=m)
    axes[1].plot(s["horizon"], s["onset recall"], "-o", color=c, lw=3 if "proposed" in m else 1.3, label=m)
for ax, t in zip(axes, ["RMSE (mg/L) on 2025 — lower is better", "onset recall on 2025 — higher is better"]):
    ax.set_xticks(HORIZONS_TO_TRAIN); ax.set_xlabel("horizon (h)"); ax.set_title(t)
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(OUT_RESULTS / "final_comparison.png", bbox_inches="tight"); plt.show()

### Check against the frozen run

This notebook follows the same chain the project calls **stage 1** (base models fitted on 2023, meta-learner fitted
on 2024). The frozen numbers for that chain are in `results/test/metrics_*_stage1.csv`. Small differences are
expected — GPU LSTM training is not bit-for-bit reproducible, the frozen system averaged 3 seeds, and its τ was chosen
on cross-fitted 2024 predictions — but the picture should be the same.

In [ ]:
ref_r, ref_c = (ROOT / "results/test/metrics_regression_stage1.csv", ROOT / "results/test/metrics_classification_stage1.csv")
if ref_r.exists() and not SMOKE_TEST:
    rr = pd.read_csv(ref_r); cc = pd.read_csv(ref_c)
    comp = []
    for h in HORIZONS_TO_TRAIN:
        for m, mine in [("lightgbm", "lightgbm"), ("bilstm", "bilstm"), ("stacking", "STACKING + WARNING RULE (proposed)")]:
            a = FINAL[(FINAL.method == mine) & (FINAL.horizon == h)].iloc[0]
            ref_rmse = rr[(rr.method == m) & (rr.horizon == h)]["rmse"].iloc[0]
            rule = "warn" if m == "stacking" else "argmax"
            ref_on = cc[(cc.method == m) & (cc.horizon == h) & (cc.rule == rule)]["onset_recall"].iloc[0]
            comp.append({"horizon": h, "model": m, "RMSE this run": a["RMSE"], "RMSE frozen": ref_rmse,
                         "onset recall this run": a["onset recall"], "onset recall frozen": ref_on})
    display(pd.DataFrame(comp).set_index(["horizon", "model"]).style.format("{:.4f}"))
else:
    print("Skipped (smoke test, or the frozen stage-1 metrics are not in results/test/).")

## 11. Using the trained system on a new hour

The saved models can forecast any new row of features. Below, the system forecasts one real 2025 hour from a station
that was about to drop below 5 mg/L.

In [ ]:
h = hh
d = test.assign(_y=y_true(test, "classification", h), _now=CLASS_NOW["test"])
cand = d.index[(d["_now"] != 0) & (d["_y"] == 0)]                    # safe now, Low in h hours
i = cand[len(cand) // 2] if len(cand) else d.index[0]
row = test.loc[i]
p_low = ENS[h]["stacking_test_proba"][i, 0]
display(Markdown(f'''
| | |
|---|---|
| station / time | **{row["station"]}**, {row["time"]:%Y-%m-%d %H:%M} |
| DO now | **{row[DO]:.2f} mg/L** ({CLASS_NAMES[int(CLASS_NOW["test"][i])]}) |
| forecast DO in {h} h | **{ENS[h]["stacking_test_reg"][i]:.2f} mg/L** |
| P(Low) in {h} h | **{p_low:.2f}**  (τ = {TAU[h]}) |
| decision | **{"⚠️ LOW-OXYGEN WARNING" if TAU[h] is not None and p_low >= TAU[h] else "no warning"}** |
| what actually happened | DO = **{y_true(test, "regression", h)[i]:.2f} mg/L** ({CLASS_NAMES[int(d.loc[i, "_y"])]}) |
| base models' DO forecasts | {", ".join(f"{m} {PRED[m][h]['test_reg'][i]:.2f}" for m in MEMBERS)} |
'''))

## 12. Summary

In [ ]:
tt = pd.Series(TIMES)
summary = pd.DataFrame({"training time (min)": [tt.filter(like=m).sum() / 60 for m in MEMBERS]}, index=MEMBERS)
display(summary.style.format("{:.1f}"))
p = FINAL[(FINAL.method.str.contains("proposed")) & (FINAL.horizon == hh)].iloc[0]
b = FINAL[(FINAL.method == "lightgbm") & (FINAL.horizon == hh)].iloc[0]
singles = FINAL[(FINAL.method.isin(MEMBERS)) & (FINAL.horizon == hh)].set_index("method")["RMSE"]
best_name, best_rmse = singles.idxmin(), singles.min()
print(f"Total training time: {tt.sum() / 60:.1f} min | models saved to {OUT_MODELS.relative_to(ROOT)}")
display(Markdown(f'''
**Proposed system on the unseen 2025 year, {hh} h ahead:** R² **{p["R2"]:.3f}**, RMSE **{p["RMSE"]:.3f} mg/L**,
accuracy **{p["accuracy"]:.3f}**, macro F1 **{p["macro F1"]:.3f}**, Low recall **{p["Low recall"]:.3f}**,
onset recall **{p["onset recall"]:.3f}** at a false-alarm rate of **{p["false-alarm rate"] * 100:.2f} %**.
Best single model (LightGBM, argmax): RMSE {b["RMSE"]:.3f}, onset recall {b["onset recall"]:.3f}.

* Five different learners were trained — three tree ensembles and two deep networks — with every setting taken from the frozen configuration.
* The **stacking meta-learner**, fitted on a year the base models never saw, gives RMSE **{p["RMSE"]:.3f}** against **{best_rmse:.3f}** for the best single model ({best_name}) — {"lower" if p["RMSE"] < best_rmse else "not lower"} than every base model.
* The **warning rule** lifts onset recall from **{b["onset recall"]:.3f}** (LightGBM, argmax) to **{p["onset recall"]:.3f}**, at a false-alarm rate of **{p["false-alarm rate"] * 100:.2f} %** ({"inside" if p["false-alarm rate"] <= 0.02 else "above"} the 2 % promise).
'''))